# **Part 1 — Python & Data Foundations: ingest, validate, clean, enrich**

###**Task 1.1 — Read the file the foundations way**



In [1]:
import csv
import json

accepted = []
rejected = []

def parse_row(row):
    # Copy row to avoid modifying original
    parsed = dict(row)
    # Cast required columns
    parsed['age'] = int(row['age'])
    parsed['fee_inr'] = int(row['fee_inr'])
    parsed['prior_no_shows'] = int(row['prior_no_shows'])
    parsed['distance_km'] = float(row['distance_km'])
    return parsed

with open('appointments.csv', newline='') as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames

    # Convert reader to list to count rows and process
    rows = list(reader)
    row_count = len(rows)

    print(f"Header: {header}")
    print(f"Total Row Count: {row_count}")

    # Loop and parse rows
    for row in rows:
        try:
            parsed_row = parse_row(row)
            accepted.append(parsed_row)
        except ValueError:
            rejected.append(row)

# Save rejected rows as rejected_rows.json
with open('rejected_rows.json', 'w') as f:
    json.dump(rejected, f, indent=2)

print(f"Successfully processed: {len(accepted)} accepted, {len(rejected)} rejected.")

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Row Count: 1020
Successfully processed: 988 accepted, 32 rejected.


#**Task 1.2 — Inspect and clean, pass 1**


In [6]:

import pandas as pd

# 1. Load the original file
df = pd.read_csv('appointments.csv')

print("INITIAL DATASET INFORMATION")
print("---------------------------")
print("Shape:", df.shape)

print("\nDataset information:")
df.info()

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nExact duplicate rows:")
print(df.duplicated().sum())

# Check distinct booking_channel values before cleaning
print("\nDistinct raw booking_channel values:")
print(df['booking_channel'].nunique())

# (a) Drop exact duplicate rows
cleaned_df = df.drop_duplicates().copy()

print("\nSTEP (a): After dropping duplicates")
print("Rows:", len(cleaned_df))

# (b) Clean booking_channel
cleaned_df['booking_channel'] = (
    cleaned_df['booking_channel'].str.strip().str.title()
)

print("\nSTEP (b): After cleaning booking_channel")
print("Rows:", len(cleaned_df))
print("Distinct cleaned booking_channel values:")
print(cleaned_df['booking_channel'].nunique())
print("Booking channel categories:")
print(cleaned_df['booking_channel'].unique())

# (c) Convert date columns
cleaned_df['appt_date'] = pd.to_datetime(
    cleaned_df['appt_date'], dayfirst=True,
    errors='coerce'
)

cleaned_df['booked_on'] = pd.to_datetime(
    cleaned_df['booked_on'], dayfirst=True,
    errors='coerce'
)

print("\nSTEP (c): After datetime conversion")
print("Rows:", len(cleaned_df))
print("Missing appt_date after conversion:",
      cleaned_df['appt_date'].isna().sum())

# (d) Drop rows with missing appt_date or status
cleaned_df = cleaned_df.dropna(
    subset=['appt_date', 'status']
).copy()

print("\nSTEP (d): After dropping missing appt_date or status")
print("Final row count:", len(cleaned_df))

# Keep the cleaned dataset for Task 1.3
cleaned1_df = cleaned_df

INITIAL DATASET INFORMATION
---------------------------
Shape: (1020, 14)

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+

#**Task 1.3 — Clean, pass 2: validity, gaps and outliers**


In [8]:

import numpy as np

# (a) Treat ages below 1 or above 100 as invalid
invalid_age_count = (
    (cleaned1_df['age'] < 1) | (cleaned1_df['age'] > 100)
).sum()

cleaned1_df.loc[
    (cleaned1_df['age'] < 1) | (cleaned1_df['age'] > 100), 'age'
] = np.nan

# Count missing ages before filling
missing_age_count = cleaned1_df['age'].isna().sum()

# (b) Fill missing age with median, round and cast to int
median_age = cleaned1_df['age'].median()

cleaned1_df['age'] = (
    cleaned1_df['age'].fillna(median_age).round().astype(int)
)

# Fill missing distance_km with median
median_distance = cleaned1_df['distance_km'].median()

cleaned1_df['distance_km'] = (
    cleaned1_df['distance_km'].fillna(median_distance)
)

# (c) Calculate Q1, Q3 and IQR for distance_km
distance_q1 = cleaned1_df['distance_km'].quantile(0.25)
distance_q3 = cleaned1_df['distance_km'].quantile(0.75)
distance_iqr = distance_q3 - distance_q1

distance_lower = distance_q1 - 1.5 * distance_iqr
distance_upper = distance_q3 + 1.5 * distance_iqr

# Calculate Q1, Q3 and IQR for fee_inr
fee_q1 = cleaned1_df['fee_inr'].quantile(0.25)
fee_q3 = cleaned1_df['fee_inr'].quantile(0.75)
fee_iqr = fee_q3 - fee_q1

fee_lower = fee_q1 - 1.5 * fee_iqr
fee_upper = fee_q3 + 1.5 * fee_iqr

# Count values that will be capped
dist_capped_count = (
    (cleaned1_df['distance_km'] < distance_lower) |
    (cleaned1_df['distance_km'] > distance_upper)
).sum()

fee_capped_count = (
    (cleaned1_df['fee_inr'] < fee_lower) |
    (cleaned1_df['fee_inr'] > fee_upper)
).sum()

# Cap outliers using np.clip
cleaned1_df['distance_km'] = np.clip(
    cleaned1_df['distance_km'], distance_lower, distance_upper
)

cleaned1_df['fee_inr'] = np.clip(
    cleaned1_df['fee_inr'], fee_lower, fee_upper
)

# (d) Round distance to 1 decimal and fee to whole rupees
cleaned1_df['distance_km'] = cleaned1_df['distance_km'].round(1)
cleaned1_df['fee_inr'] = cleaned1_df['fee_inr'].round().astype(int)

# Print acceptance criteria results
print("TASK 1.3 RESULTS")
print("--------------------------")

print("Invalid ages:", invalid_age_count)
print("Missing ages before filling:", missing_age_count)
print("Median age:", median_age)
print("Median distance:", median_distance)

print("\nDistance_km details:")
print("Q1:", distance_q1)
print("Q3:", distance_q3)
print("IQR:", distance_iqr)
print("Lower fence:", distance_lower)
print("Upper fence:", distance_upper)
print("Values capped:", dist_capped_count)

print("\nFee_inr details:")
print("Q1:", fee_q1)
print("Q3:", fee_q3)
print("IQR:", fee_iqr)
print("Lower fence:", fee_lower)
print("Upper fence:", fee_upper)
print("Values capped:", fee_capped_count)
print("Maximum fee after capping:", cleaned1_df['fee_inr'].max())

TASK 1.3 RESULTS
--------------------------
Invalid ages: 0
Missing ages before filling: 0
Median age: 48.0
Median distance: 8.2

Distance_km details:
Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower fence: -7.9375
Upper fence: 25.5625
Values capped: 26

Fee_inr details:
Q1: 500.0
Q3: 787.5
IQR: 287.5
Lower fence: 68.75
Upper fence: 1218.75
Values capped: 3
Maximum fee after capping: 1219


#**Task 1.4 — Features and a NumPy Summary**


In [9]:
import pandas as pd

# Calculate lead_days using .dt.days
cleaned1_df['lead_days'] = (cleaned1_df['appt_date'] - cleaned1_df['booked_on']).dt.days

# print(cleaned1_df[['lead_days']])
cleaned1_df['no_show'] = (cleaned1_df['status'].str.lower() == 'no-show').astype(int)

print(cleaned1_df[['lead_days', 'no_show']])

count_no_show = cleaned1_df['no_show'].sum()
print(f"Number of rows with no-show: {count_no_show}")
print(f"Share of rows with no-show: {count_no_show / len(cleaned1_df):.2%}")
# Compute (a) mean (b) number and share of rows (c) z-scores WITHOUT Loops
X = cleaned1_df[["age","distance_km","fee_inr","lead_days"]].to_numpy()
# (a)
print(f"Mean of each column: {X.mean(axis=0).round(2)}")
# (b)
lead_days_mask = X[:, 3] > 14
print(f"Number of rows with lead_days > 14: {lead_days_mask.sum()}")
print(f"Share of rows with lead_days > 14: {lead_days_mask.mean():.2%}")
# (c)
distance_km_z = (X[:, 1] - X[:, 1].mean()) / X[:, 1].std()
print(f"Number of rows with |z| > 2 in distance_km: {(np.abs(distance_km_z) > 2).astype(int).sum()}.")

      lead_days  no_show
0             1        1
1            10        0
2             6        0
4             4        0
5            40        1
...         ...      ...
1015         22        0
1016          4        1
1017         11        0
1018         13        0
1019         45        1

[990 rows x 2 columns]
Number of rows with no-show: 218
Share of rows with no-show: 22.02%
Mean of each column: [ 48.17   9.5  635.71  11.82]
Number of rows with lead_days > 14: 296
Share of rows with lead_days > 14: 29.90%
Number of rows with |z| > 2 in distance_km: 53.


#**Task 1.5 — Enrich the clinics from a live API**


In [10]:
import pandas as pd
import requests
import time

# 1. Load the original clinics data
clinic_df = pd.read_csv('clinics.csv')

temp_c_list = []

# 2. Iterate through each clinic to get temperature from Open-Meteo API
for index, row in clinic_df.iterrows():
    name = row['clinic_name']
    lat = row['latitude']
    lon = row['longitude']

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None
    success = False

    # Try calling the API up to 2 times (initial attempt + 1 retry)
    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                if "hourly" in data and "temperature_2m" in data["hourly"]:
                    # Take the first value (midnight-UTC reading)
                    temp_c = data["hourly"]["temperature_2m"][0]
                    success = True
                    break
                else:
                    reason = "Missing 'hourly' or 'temperature_2m' in response"
            else:
                reason = f"HTTP Status {response.status_code}"
        except requests.exceptions.RequestException as e:
            reason = f"Request exception: {str(e)}"

        # Wait a short moment before retrying
        if not success and attempt == 0:
            time.sleep(1)

    if not success:
        print(f"Failed to enrich clinic '{name}': {reason}")
        temp_c = None

    temp_c_list.append(temp_c)

# 3. Enrich DataFrame and save to CSV
clinic_df['temp_c'] = temp_c_list
clinic_df.to_csv('clinics_enriched.csv', index=False)

# Display the final enriched dataframe
display(clinic_df)

,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,19.2
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,19.5
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,18.6
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,14.4
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,14.4
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,27.9
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,16.6
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,14.0


#**Task 1.6 — Export the clean table and write the SQL load script**


In [13]:
import pandas as pd

# Sort the cleaned data by appointment_id
cleaned1_df = cleaned1_df.sort_values(by='appointment_id')

# Ensure columns are datetime objects before using .dt accessor
cleaned1_df['booked_on'] = pd.to_datetime(cleaned1_df['booked_on'], errors='coerce')
cleaned1_df['appt_date'] = pd.to_datetime(cleaned1_df['appt_date'], errors='coerce')

# Format date columns as YYYY-MM-DD text
cleaned1_df['booked_on'] = cleaned1_df['booked_on'].dt.strftime('%Y-%m-%d')
cleaned1_df['appt_date'] = cleaned1_df['appt_date'].dt.strftime('%Y-%m-%d')

# Keep exactly these 16 columns in this order
ordered_cols = [
    'appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date',
    'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel',
    'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show'
]
appointments_clean = cleaned1_df[ordered_cols]

# Validations: check uniqueness of appointment_id and that no columns have missing values
is_unique = appointments_clean['appointment_id'].is_unique
missing_counts = appointments_clean.isnull().sum().sum()

print(f"Is appointment_id unique? {is_unique}")
print(f"Total missing values in 16 columns: {missing_counts}")

# Save to appointments_clean.csv
appointments_clean.to_csv('appointments_clean.csv', index=False)
print("Saved appointments_clean.csv")

# Write load_data.sql containing INSERT statements
clinics_enriched = pd.read_csv('clinics_enriched.csv')

with open('load_data.sql', 'w') as f:
    # 1. Write clinic INSERT statements
    for _, row in clinics_enriched.iterrows():
        # If temp_c is missing, write NULL without quotes, else write the numeric value
        temp_val = 'NULL' if pd.isna(row['temp_c']) else str(row['temp_c'])
        sql = f"INSERT INTO clinics VALUES ('{row['clinic_id']}', '{row['clinic_name']}', '{row['city']}', '{row['tier']}', {row['latitude']}, {row['longitude']}, {temp_val});\n"
        f.write(sql)

    # 2. Write appointment INSERT statements
    for _, row in appointments_clean.iterrows():
        sql = f"INSERT INTO appointments VALUES ('{row['appointment_id']}', '{row['patient_id']}', '{row['clinic_id']}', '{row['booked_on']}', '{row['appt_date']}', {row['age']}, '{row['gender']}', {row['distance_km']}, {row['fee_inr']}, '{row['booking_channel']}', '{row['visit_type']}', '{row['reminder_sent']}', {row['prior_no_shows']}, '{row['status']}', {row['lead_days']}, {row['no_show']});\n"
        f.write(sql)

print("\nSuccessfully created load_data.sql !!")

Is appointment_id unique? True
Total missing values in 16 columns: 0
Saved appointments_clean.csv

Successfully created load_data.sql !!
